If its the first time you are running webscraping you will need to install the below libs. Just remove the #'s'

In [1]:
#pip intall fuzzywuzzy
#pip install BeautifulSoup
#pip install requests

In [2]:
from bs4 import BeautifulSoup
import requests, os, csv, urllib.parse, urllib3, time, random
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

import numpy as np
import pandas as pd

import re,os,glob, time,string
urllib3.disable_warnings()

import web_general
from IPython.display import Image

In [24]:
df = pd.DataFrame(columns=["Company_Name","Location","Revenue($M)","Employees","Summary","Year_est","Industries","Page"])

Enter the first page URL

In [19]:
URL='https://www.dnb.com/business-directory/company-information.utilities.za.html?page=1'

#Dont change the base
base_url="https://www.dnb.com/"

In [20]:
soup=web_general.parse_listing(web_general.buildSearchUrl(URL), web_general.get_proxies())
if not soup:print("Not found")
else: print("good to go")

good to go


The code is set up to loop through all the pages until we return to page 1 again or the same page repeats. As a result the last
few enteries are repeated. If you can think of a better stopping criteria let me know please

Enter the URL without the page number in the below code - see comment

In [25]:
#Enter the url link without the page number
next_page_url='https://www.dnb.com/business-directory/company-information.utilities.za.html?page='

#Change as needed
file_name="test"

In [28]:
page=2
soup=web_general.parse_listing(web_general.buildSearchUrl(URL), web_general.get_proxies())
showing_per_page=soup.find_all("div", class_="col-md-12 data")

while True:
    #Current page
    for row in range(len(showing_per_page)):
        get_name=showing_per_page[row ].find("div", class_="col-md-6").text.split()
        name=" ".join(get_name)
        print(name,": Page",page-1)
        get_location=showing_per_page[row ].find("div", class_="col-md-4").text.split()
        location=" ".join(get_location[1:])  #Remove country tag
        try:
            get_hyperlink=showing_per_page[row ].find('a').get('href')
        except:
            get_hyperlink=None

        if get_hyperlink:

            new_url=base_url+get_hyperlink
            soup_2=web_general.parse_listing(web_general.buildSearchUrl(new_url), web_general.get_proxies())

            company_summary=soup_2.find_all("span", class_="company_summary")[0].text
            try:
                industry_list=[]
                for item in range(len(soup_2.find_all("span", class_="profile-industry-item"))):
                    indus=soup_2.find_all("span", class_="profile-industry-item")[item].text.split()
                    indus=" ".join(indus)
                    industry_list.append(indus)
            except:
                industry_list=["NA"]

            industry=", ".join(industry_list)

            try:
                num_emp=(soup_2.find_all("li", class_="empCon")[0].text.split())[-1]
            except:
                num_emp="Not avaliable"

            try:
                rev=(soup_2.find_all("li", class_="sales")[0].text.split())[-1]
            except:
                rev="Not avaliable"

            try:
                year_est=(soup_2.find_all("li", class_="year_started")[0].text.split())[-1]
            except:
                year_est="Not avaliable"
            
            #Save all of the infomation into a dataframe
            dict_={"Company_Name":name,"Location":location,"Revenue($M)":rev,"Employees":num_emp,"Summary":company_summary
                   ,"Year_est":year_est,"Industries":industry,"Page":page-1}
            df = df.append(dict_, ignore_index = True)
    
    #next page
    next_page=next_page_url+str(page)
    #soup file
    soup_next_page=web_general.parse_listing(web_general.buildSearchUrl(next_page), web_general.get_proxies())

    if soup_next_page:
        get_name=showing_per_page[0].find("div", class_="col-md-6").text.split()
        get_name=showing_per_page[0].find("div", class_="col-md-6").text.split()
        name=" ".join(get_name)
        print("Current_Page",name,page-1)
         
        showing_per_page_new=soup_next_page.find_all("div", class_="col-md-12 data")
        get_name_new=showing_per_page_new[0].find("div", class_="col-md-6").text.split()
        name_new=" ".join(get_name_new)
        print("Next page",name_new,page)

        if name_new==name:
            break
        else:
            showing_per_page=showing_per_page_new
            page+=1
    print("")


df.to_csv(file_name+".csv")

SASOL SOUTH AFRICA LTD : Page 1
CITY POWER JOHANNESBURG (PTY) LTD : Page 1
THE PETROLEUM OIL AND GAS CORPORATION OF SOUTH AFRICA (PTY) : Page 1
CRYOGAS EXPRESS (PTY) LTD : Page 1
CBI-ELECTRIC TELECOM CABLES (PTY) LTD : Page 1
STAR POINT ELECTRICAL CC : Page 1
B AND W INSTRUMENTATION AND ELECTRICAL LTD : Page 1
KELVIN POWER (PTY) LTD : Page 1
SPILL TECH HIRE (PTY) LTD : Page 1
SLMTK INVESTMENTS (PTY) LTD : Page 1
BEIJER REF AFRICA (PTY) LTD : Page 1
CIVTEK (PTY) LTD : Page 1
LEPELLE NORTHERN WATER : Page 1
PROXA (PTY) LTD : Page 1
PREMIER RAINWATERGOODS (PTY) LTD : Page 1
SVA ELECTRONICS (PTY) LTD : Page 1
ENERGY PARTNERS HOLDINGS (PTY) LTD : Page 1
ZIZAMELE ELECTRICAL AND GENERAL MAINTENANCE CC : Page 1
UNICORN CHROME (PTY) LTD : Page 1
ABLON CONSTRUCTION CC : Page 1
TALBOT AND TALBOT (PTY) LTD : Page 1


KeyboardInterrupt: 

In [ ]:
12:33-13:30

In [27]:
df

,Company_Name,Location,Revenue($M),Employees,Summary,Year_est,Industries,Page
0,SASOL SOUTH AFRICA LTD,"Gauteng, Gauteng, South Africa","4,989.96",Not avaliable,"SASOL SOUTH AFRICA LTD is located in GAUTENG, ...",1968,Other Chemical Product and Preparation Manufac...,1
1,CITY POWER JOHANNESBURG (PTY) LTD,"Johannesburg, Gauteng, South Africa",253.33,"2,000",CITY POWER JOHANNESBURG (PTY) LTD is located i...,2000,"Electric Power Generation, Transmission and Di...",1
2,THE PETROLEUM OIL AND GAS CORPORATION OF SOUTH...,"Cape Town, Western Cape, South Africa",206.06,"1,588",THE PETROLEUM OIL AND GAS CORPORATION OF SOUTH...,1970,"Petroleum and Coal Products Manufacturing, Sup...",1
3,CRYOGAS EXPRESS (PTY) LTD,"Gauteng, Gauteng, South Africa",173.25,30,CRYOGAS EXPRESS (PTY) LTD is located in GAUTEN...,2008,"Other Miscellaneous Store Retailers, Electric ...",1
4,CBI-ELECTRIC TELECOM CABLES (PTY) LTD,"Gauteng, Gauteng, South Africa",70.71,500,CBI-ELECTRIC TELECOM CABLES (PTY) LTD is locat...,1967,Household Appliances and Electrical and Electr...,1
5,STAR POINT ELECTRICAL CC,"Halfway House, Gauteng, South Africa",63.33,500,STAR POINT ELECTRICAL CC is located in HALFWAY...,1997,"Electric Power Generation, Transmission and Di...",1


In [13]:
file_name="Manufacturing"
df.to_csv(file_name+".csv")